<a href="https://colab.research.google.com/github/Rodrigoldarocha/analise-chamados-python/blob/main/analise_chamados_python_std.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# -*- coding: utf-8 -*-
"""
Pipeline STD - Tratamento, Métricas de SLA, Análises e Exportação Excel
Versão para Google Colab
"""

import logging
import warnings
import calendar
from pathlib import Path
from typing import Dict, Tuple
import numpy as np
import pandas as pd
from openpyxl.styles import PatternFill

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    files = None

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)


# ======================
# CONFIGURAÇÕES
# ======================
class Config:
    DIVISOES = {
        "DIV 01": {"GO": ["GO 01"],         "UFs": ["AL", "CE", "PB", "PE", "RN"]},
        "DIV 02": {"GO": ["GO 01", "GO 02"],"UFs": ["BA", "SE"]},
        "DIV 03": {"GO": ["GO 01"],         "UFs": ["CE", "MA", "PI"]},
        "DIV 04": {"GO": ["GO 02"],         "UFs": ["AP", "PA"]},
        "DIV 05": {"GO": ["GO 02"],         "UFs": ["AM", "RO", "RR", "AC"]},
        "DIV 06": {"GO": ["GO 02"],         "UFs": ["DF", "GO"]},
        "DIV 07": {"GO": ["GO 02"],         "UFs": ["MT", "MS"]},
        "DIV 08": {"GO": ["GO 03"],         "UFs": ["SP"]},
        "DIV 09": {"GO": ["GO 03"],         "UFs": ["RJ", "MG"]},
        "DIV 10": {"GO": ["GO 03"],         "UFs": ["ES", "PR", "SC", "RS"]},
    }

    COLUNAS_IMPORTANTES = [
        "Divisão", "Gerência Operacional", "UF", "Base", "Origem", "Classificacao",
        "Solicitante", "Data_Criacao", "Data_Chegada", "Data_Previsao_Conclusao",
        "Data_Previsao_Chegada", "Data_Conclusao", "Data_de_Fechamento", "Fila",
        "Grupo", "Numero_Chamado", "Prioridade", "Substatus", "Status", "SubTipo",
        "Tipo", "Valor_Total", "Negocio", "Local_Nome", "Uniorg_Comercial",
        "Tempo_de_Custo", "Data_do_Primeiro_Encaminhamento", "Fornecedor",
        "Nota_Inicial", "originador", "regional", "Responsavel", "prazo_inicio",
        "prazo_conclusao", "rede", "modulo", "DURAÇÃO CHAMADO", "Duração Chamado",
    ]

    DATE_COLUMNS = [
        "Data_Criacao", "Data_Chegada", "Data_Previsao_Conclusao",
        "Data_Previsao_Chegada", "Data_Conclusao", "Data_de_Fechamento",
        "Data_do_Primeiro_Encaminhamento",
    ]

    META_SLA     = 96.0
    META_LIMPEZA = 98.0

    RED_FILL    = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
    GREEN_FILL  = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
    YELLOW_FILL = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
    ORANGE_FILL = PatternFill(start_color="FFD966", end_color="FFD966", fill_type="solid")

    BASE_DIR  = Path("/content")
    UPLOAD_DIR = BASE_DIR / "uploads"
    OUTPUT_DIR = BASE_DIR / "output"

    CSV_FILENAME            = "PreventivasFornecedor.csv"
    FILE_PATH               = UPLOAD_DIR / CSV_FILENAME
    OUTPUT_BASE_TRATADA     = OUTPUT_DIR / "Base_Tratada.xlsx"
    OUTPUT_ANALISE_COMPLETA = OUTPUT_DIR / "Analise_Chamados_Completa.xlsx"

    TOP_RESPONSABLES         = 15
    DIAS_FECHAMENTO_PENDENTE = 30


# ======================
# UTILITÁRIOS
# ======================
class DateUtils:
    @staticmethod
    def business_days_between(start_date, end_date) -> int:
        if pd.isna(start_date) or pd.isna(end_date) or start_date.date() == end_date.date():
            return 0
        return max(0, np.busday_count(start_date.date(), end_date.date()))

    @staticmethod
    def format_time_duration(seconds: float) -> str:
        if pd.isna(seconds):
            return "00:00:00"
        h, rem = divmod(int(seconds), 3600)
        m, s   = divmod(rem, 60)
        return f"{h:02d}:{m:02d}:{s:02d}"


# ======================
# COLAB HELPERS
# ======================
def setup_colab_environment():
    Config.UPLOAD_DIR.mkdir(exist_ok=True)
    Config.OUTPUT_DIR.mkdir(exist_ok=True)
    logger.info(f"Ambiente configurado — uploads: {Config.UPLOAD_DIR} | saída: {Config.OUTPUT_DIR}")


def upload_file_to_colab():
    if not IN_COLAB:
        logger.error("Função exclusiva do Google Colab")
        return None

    print("=" * 60)
    print(f"FAÇA O UPLOAD DO ARQUIVO CSV — esperado: {Config.CSV_FILENAME}")
    print("=" * 60)

    uploaded = files.upload()
    if not uploaded:
        logger.error("Nenhum arquivo enviado")
        return None

    for filename, content in uploaded.items():
        file_path = Config.UPLOAD_DIR / filename
        file_path.write_bytes(content)
        logger.info(f"Arquivo salvo: {file_path}")
        if filename != Config.CSV_FILENAME:
            file_path = file_path.rename(Config.UPLOAD_DIR / Config.CSV_FILENAME)
            logger.info(f"Renomeado para: {Config.CSV_FILENAME}")
        return file_path

    return None


def check_file_exists() -> bool:
    return Config.FILE_PATH.exists()


# ======================
# PROCESSAMENTO
# ======================
class STDDataProcessor:
    def __init__(self, file_path: str):
        self.file_path    = Path(file_path)
        self.df_original  = pd.DataFrame()
        self.df_processed = pd.DataFrame()
        self.stats: Dict  = {}
        self.calendario   = pd.DataFrame()
        # Mapeamentos UF -> Divisão / GO
        self.uf_para_divisao = {}
        self.uf_para_go      = {}
        for divisao, info in Config.DIVISOES.items():
            for uf in info["UFs"]:
                self.uf_para_divisao[uf] = divisao
                self.uf_para_go[uf]      = info["GO"][0] if info["GO"] else "GO Não Definido"

    def load_data(self) -> pd.DataFrame:
        if not self.file_path.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {self.file_path}")
        logger.info(f"Lendo base: {self.file_path}")

        for enc in ['utf-8', 'latin1', 'iso-8859-1', 'cp1252']:
            for sep in [',', ';', '\t', '|']:
                try:
                    df = pd.read_csv(self.file_path, encoding=enc, sep=sep, low_memory=False)
                    if len(df.columns) > 1:
                        logger.info(f"Lido com encoding={enc}, sep='{sep}' — {len(df)} registros")
                        self.df_original = df
                        return df
                except Exception:
                    continue

        # Fallback com detecção automática
        self.df_original = pd.read_csv(self.file_path, encoding='utf-8-sig', sep=None, engine='python')
        logger.info(f"Lido via fallback — {len(self.df_original)} registros")
        return self.df_original

    def prepare_data(self) -> pd.DataFrame:
        if self.df_original.empty:
            raise ValueError("Dados não carregados. Use load_data().")
        logger.info("Preparando dados...")

        # Garantir que todas as colunas existam
        for col in Config.COLUNAS_IMPORTANTES:
            if col not in self.df_original.columns:
                self.df_original[col] = None

        self.df_processed = self.df_original[
            [c for c in Config.COLUNAS_IMPORTANTES if c in self.df_original.columns]
        ].copy()

        # Mapeamentos UF
        if "UF" in self.df_processed.columns:
            self.df_processed["Divisão"]              = self.df_processed["UF"].map(self.uf_para_divisao).fillna("Divisão Não Definida")
            self.df_processed["Gerência Operacional"] = self.df_processed["UF"].map(self.uf_para_go).fillna("GO Não Definida")
        else:
            logger.warning("Coluna UF não encontrada — mapeamento ignorado")

        # Limpar Valor_Total
        if "Valor_Total" in self.df_processed.columns:
            self.df_processed["Valor_Total"] = pd.to_numeric(
                self.df_processed["Valor_Total"].astype(str)
                    .str.replace('.', '', regex=False)
                    .str.replace(',', '.', regex=False),
                errors='coerce',
            )

        # Converter datas
        for col in Config.DATE_COLUMNS:
            if col in self.df_processed.columns:
                self.df_processed[col] = pd.to_datetime(self.df_processed[col], errors='coerce', dayfirst=True)
                logger.info(f"{col}: {self.df_processed[col].notna().sum()} datas convertidas")

        self._create_calendar()
        self._create_dax_equivalent_columns()
        self._identify_late_and_open_calls()
        self._calculate_sla_status()
        logger.info("Dados preparados com sucesso.")
        return self.df_processed

    def _create_calendar(self):
        if "Data_Criacao" not in self.df_processed.columns:
            return
        dt_min, dt_max = self.df_processed["Data_Criacao"].min(), self.df_processed["Data_Criacao"].max()
        if pd.isna(dt_min) or pd.isna(dt_max):
            return

        cal = pd.DataFrame({"Date": pd.date_range(dt_min, dt_max)})
        cal["Ano"]        = cal["Date"].dt.year
        cal["Mes"]        = cal["Date"].dt.month
        cal["Nome_Mes"]   = cal["Date"].dt.month_name()
        cal["Dia_Semana"] = cal["Date"].dt.day_name()
        cal["Dia_Util"]   = np.where(cal["Dia_Semana"].isin(["Saturday", "Sunday"]), "N", "S")
        cal["Semana_Ano"] = cal["Date"].dt.isocalendar().week
        cal["Mes_Ano"]    = cal["Date"].dt.to_period("M").astype(str)
        self.calendario   = cal

    def _create_dax_equivalent_columns(self):
        df   = self.df_processed
        hoje = pd.to_datetime("today").normalize()

        # Prazo ajustado (padrão NP se coluna ausente ou valor == NA)
        for col_src, col_dst in [("prazo_inicio", "Prazo_Inicio_Ajustado"), ("prazo_conclusao", "Prazo_Conclusao_Ajustado")]:
            if col_src in df.columns:
                df[col_dst] = np.where(df[col_src].astype(str).str.upper() == "NA", "NP", df[col_src])
            else:
                df[col_dst] = "NP"
                logger.warning(f"Coluna {col_src} não encontrada — padrão 'NP'")

        df["Status_Chamado"]    = np.where(df["Data_Conclusao"].isna(), "Pendente", "Concluído")
        df["Status_Fechamento"] = np.where(df["Data_Conclusao"].isna() & df["Data_de_Fechamento"].isna(), "Pendente", "Concluído")
        df["Status_Financeiro"] = np.where(df["Data_de_Fechamento"].isna(), "Pendente", "Fechado")
        df["Estoque_Atual"]     = np.where(df["Data_de_Fechamento"].isna() & df["Data_Conclusao"].isna(), "Estoque Atual", "Não")

        df["Data_Conclusao_Ajustada"] = np.where(
            df["Data_de_Fechamento"].notna() & df["Data_Conclusao"].isna(),
            df["Data_de_Fechamento"], df["Data_Conclusao"],
        )
        df["Data_Estoque"] = np.where(df["Estoque_Atual"] == "Estoque Atual", hoje, pd.NaT)

        df["Fechamento_Pendente"] = np.where(
            df["Data_de_Fechamento"].isna() & df["Data_Conclusao"].notna() &
            ((hoje - df["Data_Conclusao"]).dt.days > Config.DIAS_FECHAMENTO_PENDENTE),
            "Sim", "Não",
        )

        df["DURACAO_CHAMADO"]         = (hoje - df["Data_Criacao"]).dt.days
        df["Dias_Atrasos"]            = np.where(df["Data_Previsao_Conclusao"].notna(), (hoje - df["Data_Previsao_Conclusao"]).dt.days + 1, 0)
        df["Horas_Chegada_x_Criacao"] = np.where(df["Data_Chegada"].notna(), (df["Data_Chegada"] - df["Data_Criacao"]).dt.total_seconds(), 0)

        # Helper: dias entre datas (mínimo 1 quando existe, 0 quando ausente)
        def _days_min1(end_col, start_col):
            days = np.where(df[end_col].notna(), (df[end_col] - df[start_col]).dt.days, 0)
            return np.where(days == 0, 1, days)

        df["Dias_Chegada"]  = _days_min1("Data_Chegada",       "Data_Criacao")
        df["Dias_Conclusao"]= _days_min1("Data_Conclusao",     "Data_Criacao")
        df["Dias_Fechados"] = _days_min1("Data_de_Fechamento", "Data_Criacao")

        df["Tempo_Atendimento"] = np.where(
            df["Data_Conclusao"].notna(), (df["Data_Conclusao"] - df["Data_Criacao"]).dt.days, 0
        )

        df["Faixa_Dias_em_Aberto"] = np.select(
            [
                (df["DURACAO_CHAMADO"] > 90) & df["Data_Conclusao"].notna(),
                (df["DURACAO_CHAMADO"] > 60) & df["Data_Conclusao"].notna(),
                (df["DURACAO_CHAMADO"] > 30) & df["Data_Conclusao"].notna(),
            ],
            ["+90 dias", "+60 dias", "+30 dias"],
            default="-30 dias",
        )

        df["A_VENCER_WTM_30_DIAS"] = np.where(
            (df["DURACAO_CHAMADO"] < 30) & (df["DURACAO_CHAMADO"] >= 20),
            "À VENCER WTM +30 DIAS", "OUTROS",
        )
        df["UF_Mapa"] = df["UF"].astype(str) + "-Brasil"

        if "Data_Criacao" in df.columns:
            df["Mes_Ano"]  = df["Data_Criacao"].dt.to_period("M")
            df["Ano"]      = df["Data_Criacao"].dt.year
            df["Mes"]      = df["Data_Criacao"].dt.month
            df["Nome_Mes"] = df["Mes_Ano"].apply(
                lambda p: calendar.month_name[p.month] if hasattr(p, 'month') else str(p)
            )

    def _identify_late_and_open_calls(self):
        df   = self.df_processed
        hoje = pd.to_datetime("today").normalize()

        m_conc  = df["Data_Conclusao"].notna() & df["Data_Previsao_Conclusao"].notna() & (df["Data_Conclusao"]          > df["Data_Previsao_Conclusao"])
        m_atr   = df["Data_Conclusao"].isna()  & df["Data_Previsao_Conclusao"].notna() & (df["Data_Previsao_Conclusao"] < hoje)
        m_sem   = df["Data_Conclusao"].isna()  & df["Data_Previsao_Conclusao"].isna()
        m_ok    = ~(m_conc | m_atr | m_sem)

        df["Status_Atraso"] = np.select(
            [m_conc, m_atr, m_sem, m_ok],
            ["Concluído com Atraso", "Atrasado", "Em Aberto (Sem Previsão)", "Em Dia"],
            default="Status Indefinido",
        )

        df["Dias_Atraso"] = 0
        df.loc[m_conc, "Dias_Atraso"] = (df["Data_Conclusao"]          - df["Data_Previsao_Conclusao"]).dt.days
        df.loc[m_atr,  "Dias_Atraso"] = (hoje                          - df["Data_Previsao_Conclusao"]).dt.days
        df.loc[m_sem,  "Dias_Atraso"] = (hoje                          - df["Data_Criacao"]).dt.days

    def _calculate_sla_status(self):
        df = self.df_processed

        req_inicio    = {"Data_Previsao_Chegada", "Data_do_Primeiro_Encaminhamento", "Data_Chegada"}
        req_conclusao = {"Data_Previsao_Conclusao", "Data_Conclusao"}

        if req_inicio.issubset(df.columns):
            data_inicio = df["Data_do_Primeiro_Encaminhamento"].fillna(df["Data_Chegada"])
            df["Status_Prazo_Inicio"] = np.where(
                df["Data_Previsao_Chegada"].isna() | data_inicio.isna(), "Não Definido",
                np.where(data_inicio <= df["Data_Previsao_Chegada"], "NP", "FP"),
            )
            df["Dias_Atraso_Inicio"] = np.where(
                df["Status_Prazo_Inicio"] == "FP",
                (df["Data_Chegada"] - df["Data_Previsao_Chegada"]).dt.days, 0,
            )
        else:
            df["Status_Prazo_Inicio"] = "Não Definido"
            logger.warning("Colunas para Status_Prazo_Inicio não encontradas")

        if req_conclusao.issubset(df.columns):
            df["Status_Prazo_Conclusao"] = np.where(
                df["Data_Previsao_Conclusao"].isna(), "Não Definido",
                np.where(df["Data_Conclusao"].isna(), "Pendente",
                         np.where(df["Data_Conclusao"] <= df["Data_Previsao_Conclusao"], "NP", "FP")),
            )
            df["Dias_Atraso_Conclusao"] = np.where(
                df["Status_Prazo_Conclusao"] == "FP",
                (df["Data_Conclusao"] - df["Data_Previsao_Conclusao"]).dt.days, 0,
            )
        else:
            df["Status_Prazo_Conclusao"] = "Não Definido"
            logger.warning("Colunas para Status_Prazo_Conclusao não encontradas")

        if "Data_Criacao" in df.columns:
            now = pd.Timestamp.now()
            df["Duracao_Chamado_Dias_Uteis"] = df.apply(
                lambda x: DateUtils.business_days_between(
                    x["Data_Criacao"],
                    x["Data_Conclusao"] if pd.notna(x.get("Data_Conclusao")) else now,
                ), axis=1,
            )

    def save_processed_data(self, output_path: str):
        if self.df_processed.empty:
            raise ValueError("Sem dados processados para salvar.")
        Path(output_path).parent.mkdir(exist_ok=True)
        self.df_processed.to_excel(output_path, index=False, engine="openpyxl")
        logger.info(f"Base tratada salva: {output_path}")


# ======================
# ANÁLISE
# ======================
class STDAnalyzer:
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.results: Dict[str, pd.DataFrame] = {}

    # Helpers internos para lidar com colunas opcionais
    def _count(self, col: str, value=None) -> int:
        if col not in self.df.columns:
            return 0
        return (self.df[col] == value).sum() if value is not None else self.df[col].notna().sum()

    def _mean(self, col: str) -> float:
        return self.df[col].mean() if col in self.df.columns else 0

    def _sum(self, col: str) -> float:
        return self.df[col].sum() if col in self.df.columns else 0

    def _nunique(self, col: str) -> int:
        return self.df[col].nunique() if col in self.df.columns else 0

    def calculate_general_stats(self) -> Dict:
        df = self.df
        total         = len(df)
        total_termino = self._count("Data_Conclusao")
        np_conclusao  = self._count("Prazo_Conclusao_Ajustado", "NP")
        np_inicio     = self._count("Prazo_Inicio_Ajustado",    "NP")
        sla_inicio    = np_inicio    / total          * 100 if total          > 0 else 0
        sla_termino   = np_conclusao / total_termino  * 100 if total_termino  > 0 else 0

        chamados_atrasados = (
            df["Status_Atraso"].isin(["Atrasado", "Concluído com Atraso"]).sum()
            if "Status_Atraso" in df.columns else 0
        )

        stats = {
            "Total Chamados":                  total,
            "Total Chamados Termino":          total_termino,
            "Total Conclusão NP":              np_conclusao,
            "Total Inicio NP":                 np_inicio,
            "SLA Início":                      sla_inicio,
            "SLA Término":                     sla_termino,
            "Comparação Meta Inicio":          sla_inicio  - Config.META_SLA,
            "Comparação Meta Término":         sla_termino - Config.META_SLA,
            "Comparação Meta Limpeza Término": sla_termino - Config.META_LIMPEZA,
            "Total Estoque":                   self._count("Estoque_Atual",         "Estoque Atual"),
            "Total Fornecedor":                self._nunique("Fornecedor"),
            "Total Chamados Concluídos":       total_termino,
            "Total Chamados FP":               total - np_conclusao,
            "Fechamento Pendente":             self._count("Fechamento_Pendente",   "Sim"),
            "À VENCER WTM 30 DIAS":            self._count("A_VENCER_WTM_30_DIAS", "À VENCER WTM +30 DIAS"),
            "Chamados Atrasados":              chamados_atrasados,
            "Chamados Em Aberto":              self._count("Status_Atraso",         "Em Aberto (Sem Previsão)"),
            "Media Dias Atrasos":              self._mean("Dias_Atrasos"),
            "Média Dias Chegada":              self._mean("Dias_Chegada"),
            "Média Dias Conclusão":            self._mean("Dias_Conclusao"),
            "Média Dias Fechamento":           self._mean("Dias_Fechados"),
            "Media Tempo Atendimento":         self._mean("Tempo_Atendimento"),
            "Média Valor OS":                  self._mean("Valor_Total"),
            "Qtd Agencias":                    self._nunique("Uniorg_Comercial"),
            "Total Valor OS":                  self._sum("Valor_Total"),
            "Media Tempo Chegada":             DateUtils.format_time_duration(self._mean("Horas_Chegada_x_Criacao")),
            "Tempo Chegada":                   DateUtils.format_time_duration(self._sum("Horas_Chegada_x_Criacao")),
        }

        self.results["estatisticas_gerais"] = pd.DataFrame.from_dict(stats, orient='index', columns=['Valor'])
        return stats

    def analyze_by_dimension(self, dimension: str, top_n: int = 20) -> pd.DataFrame:
        required = {"Numero_Chamado", "Prazo_Inicio_Ajustado", "Prazo_Conclusao_Ajustado"}
        if dimension not in self.df.columns or not required.issubset(self.df.columns):
            logger.warning(f"Dimensão ou colunas necessárias não encontradas: {dimension}")
            return pd.DataFrame()

        has_dur   = "Duracao_Chamado_Dias_Uteis" in self.df.columns
        has_valor = "Valor_Total" in self.df.columns

        analysis = (
            self.df.groupby(dimension)
            .agg(
                Total_Chamados        =('Numero_Chamado', 'count'),
                NP_Inicio             =('Prazo_Inicio_Ajustado',    lambda x: (x == 'NP').sum()),
                NP_Conclusao          =('Prazo_Conclusao_Ajustado', lambda x: (x == 'NP').sum()),
                Tempo_Medio_Resolucao =(('Duracao_Chamado_Dias_Uteis', 'mean') if has_dur   else ('Numero_Chamado', 'count')),
                Valor_Total_OS        =(('Valor_Total', 'sum')                 if has_valor else ('Numero_Chamado', 'count')),
            )
            .round(2)
            .sort_values('Total_Chamados', ascending=False)
            .head(top_n)
        )
        analysis['% SLA Início']    = (analysis['NP_Inicio']    / analysis['Total_Chamados'] * 100).round(2)
        analysis['% SLA Conclusão'] = (analysis['NP_Conclusao'] / analysis['Total_Chamados'] * 100).round(2)
        return analysis

    def analyze_monthly_evolution(self) -> pd.DataFrame:
        required = {'Ano', 'Mes', 'Nome_Mes', 'Numero_Chamado', 'Prazo_Inicio_Ajustado', 'Prazo_Conclusao_Ajustado'}
        if not required.issubset(self.df.columns):
            logger.warning("Colunas necessárias não encontradas para análise mensal")
            return pd.DataFrame()

        monthly = (
            self.df.groupby(['Ano', 'Mes', 'Nome_Mes'])
            .agg(
                Total_Chamados=('Numero_Chamado', 'count'),
                NP_Inicio     =('Prazo_Inicio_Ajustado',    lambda x: (x == 'NP').sum()),
                NP_Conclusao  =('Prazo_Conclusao_Ajustado', lambda x: (x == 'NP').sum()),
            )
            .reset_index()
        )
        monthly['% SLA Início']    = (monthly['NP_Inicio']    / monthly['Total_Chamados'] * 100).round(2)
        monthly['% SLA Conclusão'] = (monthly['NP_Conclusao'] / monthly['Total_Chamados'] * 100).round(2)
        monthly['Período']         = monthly['Nome_Mes'] + ' ' + monthly['Ano'].astype(str)
        return monthly[['Período', 'Total_Chamados', 'NP_Inicio', 'NP_Conclusao', '% SLA Início', '% SLA Conclusão']]

    def get_top_responsibles(self, top_n: int = Config.TOP_RESPONSABLES) -> pd.DataFrame:
        if "Responsavel" not in self.df.columns:
            return pd.DataFrame()
        top = self.df['Responsavel'].value_counts().head(top_n).reset_index()
        top.columns = ['Responsavel', 'Total_Chamados']
        return top

    def get_fp_analysis(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        fp_i = self.df[self.df['Status_Prazo_Inicio']    == 'FP'].copy() if 'Status_Prazo_Inicio'    in self.df.columns else pd.DataFrame()
        fp_c = self.df[self.df['Status_Prazo_Conclusao'] == 'FP'].copy() if 'Status_Prazo_Conclusao' in self.df.columns else pd.DataFrame()
        return fp_i, fp_c

    def get_late_and_open_calls(self) -> pd.DataFrame:
        if "Status_Atraso" not in self.df.columns:
            return pd.DataFrame()
        return self.df[
            self.df["Status_Atraso"].isin(["Atrasado", "Concluído com Atraso", "Em Aberto (Sem Previsão)"])
        ].copy()

    def get_accumulated_metrics(self) -> pd.DataFrame:
        required = {'Data_Criacao', 'Numero_Chamado', 'Data_Conclusao'}
        if not required.issubset(self.df.columns):
            return pd.DataFrame()
        return (
            self.df.sort_values('Data_Criacao')
            .groupby('Data_Criacao')
            .agg(
                Acumulado_Criados  =('Numero_Chamado', 'count'),
                Acumulado_Concluidos=('Data_Conclusao', lambda x: x.notna().sum()),
            )
            .cumsum()
            .reset_index()
        )


# ======================
# EXPORTAÇÃO EXCEL
# ======================
class ExcelExporter:
    def __init__(self, output_path: str):
        self.output_path = output_path
        self.writer = None

    def export_analysis(self, processor: STDDataProcessor, analyzer: STDAnalyzer):
        logger.info("Exportando análise para Excel...")
        Path(self.output_path).parent.mkdir(exist_ok=True)

        try:
            with pd.ExcelWriter(self.output_path, engine='openpyxl') as writer:
                self.writer = writer
                stats = analyzer.calculate_general_stats()
                processor.stats = stats  # sincroniza stats no processor

                processor.df_processed.to_excel(writer, sheet_name='Dados_Processados', index=False)
                pd.DataFrame(list(stats.items()), columns=['Métrica', 'Valor']).to_excel(writer, sheet_name='Estatísticas_Gerais', index=False)
                self._create_dax_measures_sheet(stats).to_excel(writer, sheet_name='Medidas_DAX_Equivalentes', index=False)

                for dim in ['Divisão', 'regional', 'Tipo', 'Prioridade', 'Fornecedor', 'UF']:
                    df_dim = analyzer.analyze_by_dimension(dim)
                    if not df_dim.empty:
                        df_dim.to_excel(writer, sheet_name=f'Por_{dim}'[:31])

                for sheet_name, df_data in [
                    ('Evolução_Mensal',     analyzer.analyze_monthly_evolution()),
                    ('Top_Responsáveis',    analyzer.get_top_responsibles()),
                    ('Métricas_Acumuladas', analyzer.get_accumulated_metrics()),
                ]:
                    if not df_data.empty:
                        df_data.to_excel(writer, sheet_name=sheet_name, index=False)

                for sheet_name, df_fp in zip(['FP_Início', 'FP_Conclusão'], analyzer.get_fp_analysis()):
                    if not df_fp.empty:
                        df_fp.to_excel(writer, sheet_name=sheet_name, index=False)

                late = analyzer.get_late_and_open_calls()
                if not late.empty:
                    late.to_excel(writer, sheet_name='Chamados_Atrasados', index=False)

                if not processor.calendario.empty:
                    processor.calendario.to_excel(writer, sheet_name='Calendario', index=False)

                self._apply_formatting()

            logger.info(f"Análise exportada: {self.output_path}")

        except Exception as e:
            logger.error(f"Erro ao exportar Excel: {e}")
            raise

    def _create_dax_measures_sheet(self, stats: Dict) -> pd.DataFrame:
        def f(key, suffix=""):
            v = stats[key]
            return f"{v:.2f}{suffix}" if isinstance(v, float) else v

        rows = [
            ("Total Chamados",                 stats["Total Chamados"],                "COUNTA('Base WTM'[Numero_Chamado])"),
            ("Total Chamados Termino",          stats["Total Chamados Termino"],         "CALCULATE([Total Chamados], USERELATIONSHIP(...))"),
            ("Total Conclusão NP",              stats["Total Conclusão NP"],             "CALCULATE([total chamados termino], Prazo Conclusão = 'NP')"),
            ("Total Inicio NP",                 stats["Total Inicio NP"],                "CALCULATE([Total Chamados], Prazo Inicio = 'NP')"),
            ("SLA Início",                      f("SLA Início", "%"),                   "[Total Inicio NP]/[Total Chamados]"),
            ("SLA Término",                     f("SLA Término", "%"),                  "[Total Conclusão NP]/[total chamados termino]"),
            ("Comparação Meta Inicio",          f("Comparação Meta Inicio", " pp"),     "[SLA Início] - [Meta]"),
            ("Comparação Meta Término",         f("Comparação Meta Término", " pp"),    "[SLA Término] - [Meta]"),
            ("Comparação Meta Limpeza Término", f("Comparação Meta Limpeza Término", " pp"), "[SLA Término] - [Meta limpeza]"),
            ("Total Estoque",                   stats["Total Estoque"],                  "CALCULATE([Total Chamados]-[Total Chamados Concluídos])"),
            ("Total Fornecedor",                stats["Total Fornecedor"],               "DISTINCTCOUNT('Base WTM'[Fornecedor])"),
            ("Total Chamados Concluídos",       stats["Total Chamados Concluídos"],      "CALCULATE([Total Chamados], Data Conclusão <> BLANK())"),
            ("Total Chamados FP",               stats["Total Chamados FP"],              "[Total Chamados] - [Total Conclusão NP]"),
            ("À VENCER WTM 30 DIAS",            stats["À VENCER WTM 30 DIAS"],           "IF(AND(DURAÇÃO < 30, DURAÇÃO >= 20), 'À VENCER WTM +30 DIAS', 'OUTROS')"),
            ("Chamados Atrasados",              stats["Chamados Atrasados"],             "Chamados com prazo vencido ou concluídos com atraso"),
            ("Chamados Em Aberto",              stats["Chamados Em Aberto"],             "Chamados sem previsão de conclusão"),
            ("Media Dias Atrasos",              f("Media Dias Atrasos"),                "AVERAGE('Base WTM'[Dias atrasos])"),
            ("Média Dias Chegada",              f("Média Dias Chegada"),                "CALCULATE(AVERAGE([Dias Chegada]), Data_Chegada <> BLANK())"),
            ("Média Dias Conclusão",            f("Média Dias Conclusão"),              "CALCULATE(AVERAGE([Dias Conclusão]), Data_Conclusao <> BLANK())"),
            ("Média Dias Fechamento",           f("Média Dias Fechamento"),             "CALCULATE(AVERAGE([Dias Fechados]), Data_de_Fechamento <> BLANK())"),
            ("Media Tempo Atendimento",         f("Media Tempo Atendimento"),           "AVERAGE('Base WTM'[Tempo Atendimento])"),
            ("Média Valor OS",                  f("Média Valor OS"),                    "AVERAGE('Base WTM'[Valor_Total])"),
            ("Qtd Agencias",                    stats["Qtd Agencias"],                   "DISTINCTCOUNT('Base WTM'[Uniorg_Comercial])"),
            ("Total Valor OS",                  f("Total Valor OS"),                    "SUM('Base WTM'[Valor_Total])"),
            ("Media Tempo Chegada",             stats["Media Tempo Chegada"],            "Formato HHMMSS"),
            ("Tempo Chegada",                   stats["Tempo Chegada"],                  "Formato HHMMSS"),
        ]
        return pd.DataFrame(rows, columns=["Medida DAX", "Valor", "Descrição"])

    def _apply_formatting(self):
        if not self.writer:
            return
        wb = self.writer.book

        for sheet_name in ['Evolução_Mensal', 'Estatísticas_Gerais', 'Medidas_DAX_Equivalentes']:
            if sheet_name not in wb.sheetnames:
                continue
            ws = wb[sheet_name]
            for col in range(1, ws.max_column + 1):
                header = str(ws.cell(row=1, column=col).value or "")
                if '%' in header:
                    self._fmt_col(ws, col, mode='pct')
                elif 'Comparação' in header:
                    self._fmt_col(ws, col, mode='cmp')

        if 'Chamados_Atrasados' in wb.sheetnames:
            ws = wb['Chamados_Atrasados']
            status_col = next(
                (c for c in range(1, ws.max_column + 1) if ws.cell(1, c).value == 'Status_Atraso'), None
            )
            if status_col:
                color_map = {
                    'Atrasado':                 Config.RED_FILL,
                    'Concluído com Atraso':     Config.ORANGE_FILL,
                    'Em Aberto (Sem Previsão)': Config.YELLOW_FILL,
                }
                for row in range(2, ws.max_row + 1):
                    fill = color_map.get(ws.cell(row, status_col).value)
                    if fill:
                        for c in range(1, ws.max_column + 1):
                            ws.cell(row, c).fill = fill

    def _fmt_col(self, ws, col_idx: int, mode: str):
        """Aplica formatação condicional em coluna de percentual ou comparação."""
        for row in range(2, ws.max_row + 1):
            cell = ws.cell(row=row, column=col_idx)
            try:
                value = float(str(cell.value or 0).replace('%', '').replace('pp', '').strip())
            except (ValueError, TypeError):
                continue

            if mode == 'pct':
                header = str(ws.cell(1, col_idx).value or "")
                meta   = Config.META_SLA if 'SLA' in header else Config.META_LIMPEZA
                cell.fill = Config.GREEN_FILL if value >= meta else (Config.YELLOW_FILL if value >= meta - 5 else Config.RED_FILL)
            else:
                cell.fill = Config.GREEN_FILL if value >= 0 else Config.RED_FILL


# ======================
# SAÍDA / RELATÓRIO
# ======================
def download_results():
    if not IN_COLAB:
        logger.info("Arquivos salvos localmente.")
        return
    print("\n" + "=" * 60 + "\nDOWNLOAD DOS ARQUIVOS GERADOS\n" + "=" * 60)
    for nome, path in [("Base Tratada", Config.OUTPUT_BASE_TRATADA), ("Análise Completa", Config.OUTPUT_ANALISE_COMPLETA)]:
        if path.exists():
            print(f"\n{nome}: {path.name} ({path.stat().st_size / 1024:.1f} KB)")
            try:
                files.download(str(path))
                print("  ✓ Download iniciado")
            except Exception as e:
                print(f"  ✗ Erro: {e} — acesse: {path}")
        else:
            print(f"\n{nome}: arquivo não encontrado")


def show_summary(stats: Dict):
    print("\n" + "=" * 60 + "\nRESUMO DO PROCESSAMENTO\n" + "=" * 60)
    lines = [
        ("Total de chamados",    f"{stats['Total Chamados']:,.0f}"),
        ("SLA Início",           f"{stats['SLA Início']:.2f}%"),
        ("SLA Término",          f"{stats['SLA Término']:.2f}%"),
        ("Estoque atual",        f"{stats['Total Estoque']:,.0f}"),
        ("Chamados Atrasados",   f"{stats['Chamados Atrasados']:,.0f}"),
        ("Chamados Em Aberto",   f"{stats['Chamados Em Aberto']:,.0f}"),
        ("Total Valor OS",       f"R$ {stats['Total Valor OS']:,.2f}"),
        ("Total Fornecedores",   f"{stats['Total Fornecedor']:,.0f}"),
        ("Média Dias Atraso",    f"{stats['Media Dias Atrasos']:.2f}"),
        ("Média Dias Conclusão", f"{stats['Média Dias Conclusão']:.2f}"),
    ]
    for label, value in lines:
        print(f"{label}: {value}")
    print("=" * 60)


# ======================
# MAIN
# ======================
def main():
    try:
        logger.info("=" * 60 + "\nINICIANDO PROCESSAMENTO STD — GOOGLE COLAB\n" + "=" * 60)
        setup_colab_environment()

        if not check_file_exists():
            if not upload_file_to_colab():
                logger.error("Upload cancelado. Encerrando.")
                return
        else:
            logger.info(f"Arquivo encontrado: {Config.FILE_PATH}")
            if input("Deseja fazer novo upload? (s/N): ").strip().lower() == 's':
                upload_file_to_colab()

        processor = STDDataProcessor(str(Config.FILE_PATH))
        processor.load_data()
        processor.prepare_data()
        processor.save_processed_data(str(Config.OUTPUT_BASE_TRATADA))

        analyzer = STDAnalyzer(processor.df_processed)
        stats    = analyzer.calculate_general_stats()

        ExcelExporter(str(Config.OUTPUT_ANALISE_COMPLETA)).export_analysis(processor, analyzer)

        logger.info("PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
        show_summary(stats)
        download_results()

    except Exception as e:
        logger.error(f"Erro no processamento: {e}")
        import traceback
        traceback.print_exc()
        raise


if __name__ == "__main__":
    main()

Deseja fazer novo upload? (s/N): s
FAÇA O UPLOAD DO ARQUIVO CSV — esperado: PreventivasFornecedor.csv


Saving PreventivasFornecedor.csv to PreventivasFornecedor (2).csv

RESUMO DO PROCESSAMENTO
Total de chamados: 11,582
SLA Início: 90.31%
SLA Término: 91.83%
Estoque atual: 505
Chamados Atrasados: 1,333
Chamados Em Aberto: 93
Total Valor OS: R$ 8,487,926.59
Total Fornecedores: 1
Média Dias Atraso: 168.37
Média Dias Conclusão: 14.11

DOWNLOAD DOS ARQUIVOS GERADOS

Base Tratada: Base_Tratada.xlsx (4441.9 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ Download iniciado

Análise Completa: Analise_Chamados_Completa.xlsx (5979.5 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ Download iniciado
